# 01 · Вырождение методов атрибуции

**Запускается без API-ключа и без данных. Только numpy и scikit-learn.**

Утверждение, которое здесь проверяется:

> При `sampling_ratio = 0` occlusion, LIME и Shapley — строго возрастающие
> аффинные функции друг друга. Значит порядки, которые они индуцируют,
> совпадают **тождественно, на любом промпте**.

Это не «корреляция получилась высокой». Это выводится в закрытой форме,
и константы предсказываются заранее.

In [1]:
import numpy as np
from sklearn.linear_model import Ridge

np.set_printoptions(precision=6, suppress=True)

## Постановка

При `sampling_ratio = 0` библиотека перебирает только коалиции
«всё, кроме одного». Для $n$ инструментов это $n$ строк, и матрица
присутствия вырождается в

$$X = J - I$$

где $J$ — матрица из единиц. Строка $i$ — коалиция $N \setminus \{i\}$.

Берём реальные значения $v(N \setminus \{i\})$ из прогона агента
на финансовом промпте.

In [2]:
tools = ["get_weather", "get_stock_price", "calculate", "get_news"]
n = len(tools)

# v(N\{i}) — косинусная близость ответа без инструмента i к полному ответу
sim = np.array([0.299679, 0.139894, 0.706630, 0.662437])

v_N = 1.0                 # baseline, зашитый в коде библиотеки
occlusion = v_N - sim     # вклад по occlusion

X = 1 - np.eye(n)
for t, s, o in zip(tools, sim, occlusion):
    print(f"{t:<18} v(N\\{{i}})={s:.6f}   occlusion={o:.6f}")

get_weather        v(N\{i})=0.299679   occlusion=0.700321
get_stock_price    v(N\{i})=0.139894   occlusion=0.860106
calculate          v(N\{i})=0.706630   occlusion=0.293370
get_news           v(N\{i})=0.662437   occlusion=0.337563


## LIME

В коде веса считаются как `X.sum(axis=1) / n`. При $X = J - I$ сумма каждой
строки равна $n-1$, значит **все веса одинаковы**: $w = (n-1)/n$.
Взвешивание превращается в no-op.

Ridge центрирует данные, поэтому $\tilde X = J/n - I$ и
$\tilde X^\top \tilde X = I - J/n$. Вектор $\tilde y$ ортогонален единичному,
а на этом подпространстве у $\tilde X^\top \tilde X$ собственное значение 1.
Отсюда

$$\beta = (w\tilde X^\top \tilde X + \alpha I)^{-1} w \tilde X^\top \tilde y
        = -\frac{w}{w + \alpha}\,\tilde y$$

и, поскольку $y_i = v(N) - \mathrm{occ}_i$,

$$\boxed{\ \mathrm{lime}_i = \frac{w}{w+\alpha}\,\mathrm{occ}_i + c\ }$$

In [3]:
w = X.sum(axis=1) / n
alpha = 1.0

predicted_slope = w[0] / (w[0] + alpha)
print(f"веса:              {w}          (константа -> взвешивание ничего не делает)")
print(f"предсказанный наклон w/(w+alpha) = {predicted_slope:.9f}")
print(f"он же дробью:      (n-1)/(2n-1) = {n-1}/{2*n-1} = {(n-1)/(2*n-1):.9f}")

веса:              [0.75 0.75 0.75 0.75]          (константа -> взвешивание ничего не делает)
предсказанный наклон w/(w+alpha) = 0.428571429
он же дробью:      (n-1)/(2n-1) = 3/7 = 0.428571429


In [4]:
model = Ridge(alpha=alpha)
model.fit(X, sim, sample_weight=w)
lime = model.coef_

slope, intercept = np.polyfit(occlusion, lime, 1)
corr = np.corrcoef(occlusion, lime)[0, 1]

print(f"lime          = {lime}")
print(f"измеренный наклон  = {slope:.9f}")
print(f"предсказанный      = {predicted_slope:.9f}")
print(f"совпадает          = {np.isclose(slope, predicted_slope)}")
print(f"корреляция с occlusion = {corr:.12f}")

lime          = [ 0.065349  0.133828 -0.109059 -0.090119]
измеренный наклон  = 0.428571429
предсказанный      = 0.428571429
совпадает          = True
корреляция с occlusion = 1.000000000000


**Следствие.** Ridge-регуляризация здесь не делает ничего, кроме сжатия
occlusion на известный множитель. При $\alpha \to 0$ LIME совпадает с
occlusion тождественно — вся «разница между методами» это один гиперпараметр.

In [5]:
for a in [1.0, 0.1, 0.01, 1e-6]:
    m = Ridge(alpha=a); m.fit(X, sim, sample_weight=w)
    s = np.polyfit(occlusion, m.coef_, 1)[0]
    print(f"alpha={a:<8}  наклон={s:.6f}   предсказание w/(w+a)={w[0]/(w[0]+a):.6f}")

alpha=1.0       наклон=0.428571   предсказание w/(w+a)=0.428571
alpha=0.1       наклон=0.882353   предсказание w/(w+a)=0.882353
alpha=0.01      наклон=0.986842   предсказание w/(w+a)=0.986842
alpha=1e-06     наклон=0.999999   предсказание w/(w+a)=0.999999


## Shapley

Нормировка в библиотеке сдвигает минимум в ноль и делит на сумму:

$$\varphi_i = \frac{\mathrm{occ}_i - \min_j \mathrm{occ}_j}
                   {\sum_k \mathrm{occ}_k - n \min_j \mathrm{occ}_j}$$

Тоже аффинная функция от occlusion.

In [6]:
phi = (occlusion - occlusion.min()) / (occlusion.sum() - n * occlusion.min())

slope_s, intercept_s = np.polyfit(occlusion, phi, 1)
print(f"shapley       = {phi}")
print(f"сумма         = {phi.sum():.6f}   (эффективность выполняется нормировкой)")
print(f"shapley = {slope_s:.4f} * occlusion + {intercept_s:.4f}")
print(f"корреляция    = {np.corrcoef(occlusion, phi)[0,1]:.12f}")

shapley       = [0.399803 0.556781 0.       0.043417]
сумма         = 1.000000   (эффективность выполняется нормировкой)
shapley = 0.9824 * occlusion + -0.2882
корреляция    = 1.000000000000


## Вот главный дефект

Нормировка **принудительно обнуляет минимальный инструмент**. Аксиома dummy
оказывается «выполнена» ровно для одного инструмента на промпт — случайно,
по построению.

А на промптах, где не нужен вообще ни один инструмент, обнуляется один,
а остальные получают паразитную положительную массу.

In [7]:
print("ноль получает:", tools[int(np.argmin(occlusion))],
      "— просто потому, что у него минимальный occlusion\n")

# все инструменты лишние: v(N\{i}) почти одинаковы, различия — шум генерации
rng = np.random.default_rng(0)
noise_floor = 0.93                      # две генерации с полным тулсетом
sim_dummy = noise_floor + rng.normal(0, 0.01, n)
occ_d = 1.0 - sim_dummy
phi_d = (occ_d - occ_d.min()) / (occ_d.sum() - n * occ_d.min())

for t, p in zip(tools, phi_d):
    print(f"{t:<18} shapley={p: .4f}   <- должен быть 0, ни один не вызывался")

ноль получает: calculate — просто потому, что у него минимальный occlusion

get_weather        shapley= 0.2824   <- должен быть 0, ни один не вызывался
get_stock_price    shapley= 0.4238   <- должен быть 0, ни один не вызывался
calculate          shapley= 0.0000   <- должен быть 0, ни один не вызывался
get_news           shapley= 0.2938   <- должен быть 0, ни один не вызывался


## Порядки совпадают тождественно

Обе функции строго возрастают по occlusion, значит ранжирования одинаковы
не «в этом примере», а всегда.

In [8]:
rng = np.random.default_rng(1)
for trial in range(5):
    s = rng.uniform(0.05, 0.95, n)
    occ_t = 1.0 - s
    m = Ridge(alpha=1.0); m.fit(X, s, sample_weight=w)
    ph = (occ_t - occ_t.min()) / (occ_t.sum() - n * occ_t.min())
    r_occ, r_lime, r_phi = (tuple(int(j) for j in np.argsort(-x)) for x in (occ_t, m.coef_, ph))
    print(f"прогон {trial}: occlusion={r_occ}  lime={r_lime}  shapley={r_phi}"
          f"   одинаковы={r_occ == r_lime == r_phi}")

прогон 0: occlusion=(2, 0, 3, 1)  lime=(2, 0, 3, 1)  shapley=(2, 0, 3, 1)   одинаковы=True
прогон 1: occlusion=(0, 3, 1, 2)  lime=(0, 3, 1, 2)  shapley=(0, 3, 1, 2)   одинаковы=True
прогон 2: occlusion=(1, 3, 0, 2)  lime=(1, 3, 0, 2)  shapley=(1, 3, 0, 2)   одинаковы=True
прогон 3: occlusion=(2, 0, 3, 1)  lime=(2, 0, 3, 1)  shapley=(2, 0, 3, 1)   одинаковы=True
прогон 4: occlusion=(0, 2, 3, 1)  lime=(0, 2, 3, 1)  shapley=(0, 2, 3, 1)   одинаковы=True


## Что с этим делать

1. Не сравнивать методы, не проверив, не вырождены ли они на вашей сетке коалиций.
2. При $n \le 8$ считать Шепли точно: полный перебор $2^n \le 256$ коалиций
   из одного кэша $v(S)$. Приближение здесь не нужно вообще.
3. Починить baseline: `max(0.0, noise_floor - similarity)` вместо
   недостижимой единицы.

In [9]:
from itertools import combinations
from math import factorial

def exact_shapley(v, players):
    # Точные значения Шепли полным перебором. n <= 8 -> не более 256 коалиций.
    N = list(players); n = len(N)
    phi = {}
    for i in N:
        rest = [p for p in N if p != i]
        total = 0.0
        for k in range(len(rest) + 1):
            wk = factorial(k) * factorial(n - k - 1) / factorial(n)
            for S in combinations(rest, k):
                total += wk * (v(frozenset(S) | {i}) - v(frozenset(S)))
        phi[i] = total
    return phi

# игрушечный v: search_a и search_b взаимозаменяемы
def v(S):
    return (1.0 if ({"search_a"} & S or {"search_b"} & S) else 0.0) + (0.4 if "calc" in S else 0.0)

print("Теорема об избыточности на практике:")
for k, val in exact_shapley(v, ["search_a", "search_b", "calc"]).items():
    print(f"  {k:<10} phi={val: .4f}")
print("\nsearch_a и search_b делят вклад поровну, хотя PN каждого = 0:")
print("  убери любой из них — ответ не изменится.")

Теорема об избыточности на практике:
  search_a   phi= 0.5000
  search_b   phi= 0.5000
  calc       phi= 0.4000

search_a и search_b делят вклад поровну, хотя PN каждого = 0:
  убери любой из них — ответ не изменится.
